# CASE 3: Training Reproducibility with TCRβ Extension #

Since the number of models generated across the 10 folds is quite large, we provide only the Jupyter notebook for a single fold here, specifically the first fold in the reshuffling process. This notebook reproduces the zero setting of the Meta-learner model. The remaining results can be reproduced by following the same workflow for the other folds.

## classification

### PanPep

##### majority

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_majority_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"  #Blank file
!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-beta" \
    --support_dir "../data/support/majority" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE3_majority"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: LLAGIGTVPI on device: cuda:0
Positive TCRs: 324, Negative TCRs: 9552
Total batches: 1
Loading support data from: ../data/support/majority
Loading support data for peptide: LLAGIGTVPI
Query data size: 9876

Processing batch 1/1 for peptide LLAGIGTVPI - 2026-03-17 11:02:40
Finetuning model...
batch processing time: 13.27s, Progress: 100.0%
Successfully wrote 9876 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_majority/LLAGIGTVPI.parquet

Peptide LLAGIGTVPI processing time: 13.36s

Processing peptide: TTDPSFLGRY on device: cuda:0
Positive TCRs: 166, Negative TCRs: 9552
Total batches: 1
Loading support data from: ../data/support/majority
Loading support data for peptide: TTDPSFLGRY
Query data size: 9718

Processing batch 1/1 for peptide TTDPSFLGRY - 2026-03-17 11:02:53
Finetuning model...
batch processing time: 11.48s, Progress: 100.0%
Successfully wrote 9718 records to /mnt/d/PanPep_Reu

In [2]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE3_majority"
CSV_PATH = "../data/fig4/reshuffling_majority_0.csv"


OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE3_majority_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ATDALMTGF: 52 rows, 52 scored
[OK] LLAGIGTVPI: 324 rows, 324 scored
[OK] LTDEMIAQY: 52 rows, 52 scored
[OK] TTDPSFLGRY: 166 rows, 166 scored

Saved to ./result/CASE3_majority_scored.csv
Total rows: 594, scored: 594


In [4]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE3_majority_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [5]:
import pandas as pd

df = pd.read_csv("./CASE3_majority_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE3_majority_scored.csv,0.537678,0.540459,success
1,./result,[FOLDER_SUMMARY],0.537678,0.540459,success (averaged over 1 files)


##### few-shot

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_few_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_few_shot.py \
    --mode single \
    --model default\
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-beta" \
    --support_dir "../data/support/few" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE3_few"

Using 1 GPUs: [0]
Using model configuration: default
Inference mode: single
support_dir enabled: will load k-shot data from ../data/support/few

Processing peptide: NTNSSPDDQIGYY on device: cuda:0
Positive TCRs: 10, Negative TCRs: 9552
Total batches: 1
Loading k-shot data from: ../data/support/few
Query data size: 9562

Processing batch 1/1 for peptide NTNSSPDDQIGYY - 2026-03-17 14:14:41
Finetuning model...
batch processing time: 1.27s, Progress: 100.0%
Successfully wrote 9562 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_few/NTNSSPDDQIGYY.parquet

Total time for peptide NTNSSPDDQIGYY: 1.35s

Processing peptide: RLLPLLALLAL on device: cuda:0
Positive TCRs: 26, Negative TCRs: 9552
Total batches: 1
Loading k-shot data from: ../data/support/few
Query data size: 9578

Processing batch 1/1 for peptide RLLPLLALLAL - 2026-03-17 14:14:42
Finetuning model...
batch processing time: 0.60s, Progress: 100.0%
Successfully wrote 9578 records to /mnt/d/PanPep_Reusability

In [8]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE3_few"
CSV_PATH = "../data/fig4/reshuffling_few_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE3_few_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AIFYLITPV: 18 rows, 18 scored
[OK] AIIRILQQL: 12 rows, 12 scored
[OK] ALASCMGLIY: 8 rows, 8 scored
[OK] ALDIEIATY: 8 rows, 8 scored
[OK] ALSKGVHFV: 70 rows, 70 scored
[OK] ALWEIQQVV: 68 rows, 68 scored
[OK] ALWMRLLPL: 28 rows, 28 scored
[OK] AMFWSVPTV: 156 rows, 156 scored
[OK] CLAVHECFV: 16 rows, 16 scored
[OK] CLNEYHLFL: 16 rows, 16 scored
[OK] DTDFVNEFY: 12 rows, 12 scored
[OK] ELAGIGLTV: 16 rows, 16 scored
[OK] ELTSMKYFV: 6 rows, 6 scored
[OK] FIAGLIAIV: 10 rows, 10 scored
[OK] FLAHIQWMV: 14 rows, 14 scored
[OK] FLALCADSI: 10 rows, 10 scored
[OK] FLCWHTNCY: 6 rows, 6 scored
[OK] FLGKIWPSYK: 16 rows, 16 scored
[OK] FLHVTYVPA: 6 rows, 6 scored
[OK] FLLNKEMYL: 14 rows, 14 scored
[OK] FLNGSCGSV: 6 rows, 6 scored
[OK] FLNRFTTTL: 8 rows, 8 scored
[OK] FLPGVYSVI: 12 rows, 12 scored
[OK] FLPRVFSAV: 12 rows, 12 scored
[OK] FLTENLLLY: 8 rows, 8 scored
[OK] FLVLLPLVS: 6 rows, 6 scored
[OK] FLWSVFWLI: 40 rows, 40 scored
[OK] FTSDYYQLY: 76 rows, 76 scored
[OK] FTVLCLTPV: 18 rows, 18 scored

In [10]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE3_few_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [11]:
import pandas as pd

df = pd.read_csv("./CASE3_few_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE3_few_scored.csv,0.52483,0.515619,success
1,./result,[FOLDER_SUMMARY],0.52483,0.515619,success (averaged over 1 files)


#### zero-shot

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_zero_shot.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-beta" \
    --result_dir "CASE3_zero"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --model default

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: ALWMRLLPLL on device: cuda:0

Processing batch 1/1 for peptide ALWMRLLPLL - 2026-03-17 14:19:47
batch processing time: 0.96s, Progress: 100.0%
Successfully wrote 9554 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_zero/ALWMRLLPLL.parquet

Peptide ALWMRLLPLL processing time: 1.00s

Processing peptide: CTDDNALAYY on device: cuda:0

Processing batch 1/1 for peptide CTDDNALAYY - 2026-03-17 14:19:48
batch processing time: 0.60s, Progress: 100.0%
Successfully wrote 9558 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_zero/CTDDNALAYY.parquet

Peptide CTDDNALAYY processing time: 0.64s

Processing peptide: EIFDRYGEEV on device: cuda:0

Processing batch 1/1 for peptide EIFDRYGEEV - 2026-03-17 14:19:49
batch processing time: 0.62s, Progress: 100.0%
Successfully wrote 9556 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_zero/EIFDRYGEEV.pa

In [16]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE3_zero"
CSV_PATH = "../data/fig4/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE3_zero_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALPILFQV: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALLPGLPAA: 8 rows, 8 scored
[OK] ALNLGETFV: 4 rows, 4 scored
[OK] ALNNIINNA: 4 rows, 4 scored
[OK] ALPETTADI: 4 rows, 4 scored
[OK] ALWMRLLPLL: 2 rows, 2 scored
[OK] ALYYPSARI: 4 rows, 4 scored
[OK] AMDEFIERY: 4 rows, 4 scored
[OK] AMQTMLFTM: 6 rows, 6 scored
[OK] ATSRMLSYY: 2 rows, 2 scored
[OK] ATSRTLSYY: 6 rows, 6 scored
[OK] AVLDMCASL: 2 rows, 2 scored
[OK] CLAYYFMRF: 6 rows, 6 scored
[OK] CLEASFNYL: 6 rows, 6 scored
[OK] CLGSLIYST: 2 rows, 2 scored
[OK] CTDDNALAYY: 6 rows, 6 scored
[OK] CTDDNALAY: 2 rows, 2 scored
[OK] DKDPQFKDN: 2 rows, 2 scored
[OK] DLTSFLLSL: 2 rows, 2 scored
[OK] DNVILLNKH: 6 rows, 6 scored
[OK] DRLNQLESK: 2 rows, 2 scored
[OK] DYKHYTPSF: 2 rows, 2 scored
[OK] DYTEISFML: 2 rows, 2 scored
[OK] EIFDRYGEEV: 4 rows, 4 scored
[OK] ETALALLLL: 2 rows, 2 scored
[OK] EWFLAYILF: 2 rows, 2 scored
[OK] FAMQMAYRF: 8 rows, 8 scored
[OK] FFSYFAVHF: 2 rows, 2 scored
[OK] FG

In [18]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE3_zero_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [19]:
import pandas as pd

df = pd.read_csv("./CASE3_zero_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE3_zero_scored.csv,0.494368,0.491312,success
1,./result,[FOLDER_SUMMARY],0.494368,0.491312,success (averaged over 1 files)


#### Meta-learner

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_meta_learner.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-beta" \
    --result_dir "CASE3_meta"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" 

Using 1 GPUs: [0]
GPU 0 will process 230 peptides with approximately 858 TCRs

Processing peptide: ALLPGLPAA on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 9561

Processing batch 1/1 for peptide ALLPGLPAA - 2026-03-17 14:33:10
batch processing time: 1.01s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_meta/ALLPGLPAA.parquet

Total time for peptide ALLPGLPAA: 1.14s

Processing peptide: FAMQMAYRF on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 9561

Processing batch 1/1 for peptide FAMQMAYRF - 2026-03-17 14:33:11
batch processing time: 0.55s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE3_meta/FAMQMAYRF.parquet

Total time for peptide FAMQMAYRF: 0.65s

Processing peptide: FLFAVGFYL on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 956

In [22]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE3_meta"
CSV_PATH = "../data/fig4/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE3_meta_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALPILFQV: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALLPGLPAA: 8 rows, 8 scored
[OK] ALNLGETFV: 4 rows, 4 scored
[OK] ALNNIINNA: 4 rows, 4 scored
[OK] ALPETTADI: 4 rows, 4 scored
[OK] ALWMRLLPLL: 2 rows, 2 scored
[OK] ALYYPSARI: 4 rows, 4 scored
[OK] AMDEFIERY: 4 rows, 4 scored
[OK] AMQTMLFTM: 6 rows, 6 scored
[OK] ATSRMLSYY: 2 rows, 2 scored
[OK] ATSRTLSYY: 6 rows, 6 scored
[OK] AVLDMCASL: 2 rows, 2 scored
[OK] CLAYYFMRF: 6 rows, 6 scored
[OK] CLEASFNYL: 6 rows, 6 scored
[OK] CLGSLIYST: 2 rows, 2 scored
[OK] CTDDNALAYY: 6 rows, 6 scored
[OK] CTDDNALAY: 2 rows, 2 scored
[OK] DKDPQFKDN: 2 rows, 2 scored
[OK] DLTSFLLSL: 2 rows, 2 scored
[OK] DNVILLNKH: 6 rows, 6 scored
[OK] DRLNQLESK: 2 rows, 2 scored
[OK] DYKHYTPSF: 2 rows, 2 scored
[OK] DYTEISFML: 2 rows, 2 scored
[OK] EIFDRYGEEV: 4 rows, 4 scored
[OK] ETALALLLL: 2 rows, 2 scored
[OK] EWFLAYILF: 2 rows, 2 scored
[OK] FAMQMAYRF: 8 rows, 8 scored
[OK] FFSYFAVHF: 2 rows, 2 scored
[OK] FG

In [23]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE3_meta_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [24]:
import pandas as pd

df = pd.read_csv("./CASE3_meta_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE3_meta_scored.csv,0.493167,0.49455,success
1,./result,[FOLDER_SUMMARY],0.493167,0.49455,success (averaged over 1 files)


### Other method

If you would like to reproduce other methods, you can use their web servers. For DLpTCR, you can use http://jianglab.org.cn/DLpTCR/ and select pTCRβ. For ERGO-II, you can use http://tcr2.cs.biu.ac.il/home and choose VDJdb mode.

## rank

For demonstration purposes, since running on the full background pool would be too time-consuming, we constructed a subset pool by randomly sampling 0.1% of the background library and used it for reproduction and result presentation. If you would like to reproduce the full original results, you can simply replace the subset background pool with the complete background library and rerun the same pipeline. In other words, the current setup is only for demonstration efficiency, while the full results can be recovered by using the original background pool.

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/majority-fig3.csv"
NEGATIVE_DATA = "../data/Combined_library_sample_0.1pct.txt"

!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-beta" \
    --support_dir "../data/support/majority" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE3_majority_rank"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: LLAGIGTVPI on device: cuda:0
Positive TCRs: 806, Negative TCRs: 57103
Total batches: 6
Loading support data from: ../data/support/majority
Loading support data for peptide: LLAGIGTVPI
Query data size: 57265

Processing batch 1/6 for peptide LLAGIGTVPI - 2026-03-17 14:43:00
Finetuning model...
batch processing time: 13.74s, Progress: 16.7%

Processing batch 2/6 for peptide LLAGIGTVPI - 2026-03-17 14:43:14
Using finetuned model for inference...
batch processing time: 0.72s, Progress: 33.3%

Processing batch 3/6 for peptide LLAGIGTVPI - 2026-03-17 14:43:15
Using finetuned model for inference...
batch processing time: 2.88s, Progress: 50.0%

Processing batch 4/6 for peptide LLAGIGTVPI - 2026-03-17 14:43:18
Using finetuned model for inference...
batch processing time: 0.69s, Progress: 66.7%

Processing batch 5/6 for peptide LLAGIGTVPI - 2026-03-17 14:43:18
Using finetuned model for inference...
batch processing time: 

In [ ]:
!{sys.executable} ../metric_calculation/sort.py \
                    --input_dir ./CASE3_majority_rank \
                    --output_dir ./CASE3_majority_sort

Found 4 files to process:
  CSV files: 0
  Parquet files: 4

Completed processing 4 files


In [ ]:
!{sys.executable} ../metric_calculation/bedroc.py \
                    -i ./CASE3_majority_sort/CASE1_majority_rank \
                    -o ./CASE3_majority_bedroc

2026-03-17 14:47:14,068 - INFO - Input directory: /mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE3_majority_rank
2026-03-17 14:47:14,068 - INFO - Output directory: ./CASE3_majority_bedroc
2026-03-17 14:47:14,068 - INFO - Number of processes: 8
Total: 4 files, To process: 4 files
Processing files:   0%|                                   | 0/4 [00:00<?, ?it/s]
File: TTDPSFLGRY_sorted.parquet

File: LLAGIGTVPI_sorted.parquet

File: LTDEMIAQY_sorted.parquet

File: ATDALMTGF_sorted.parquet
Processing files: 100%|███████████████████████████| 4/4 [00:00<00:00, 32.20it/s]
2026-03-17 14:47:14,266 - INFO - Successfully processed 4 files.
2026-03-17 14:47:14,267 - INFO - Detailed results stored in: ./CASE3_majority_bedroc/
2026-03-17 14:47:14,267 - INFO - Starting summary calculation...
2026-03-17 14:47:14,271 - INFO - Reading 4 result files...
Reading detailed results: 100%|██████████████████| 4/4 [00:00<00:00, 183.34it/s]
2026-03-17 14:47:14,309 - INFO - Summary saved to

In [ ]:
import os
import glob
import pandas as pd

detailed_dir = "./CASE3_majority_bedroc"

# Print all CSV files in the detailed results directory
csv_files = sorted(glob.glob(os.path.join(detailed_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV file(s) in: {detailed_dir}\n")

for file_path in csv_files:
    print("=" * 100)
    print(f"File: {file_path}")
    print("=" * 100)
    df = pd.read_csv(file_path)
    print(df)
    print("\n")

Found 5 CSV file(s) in: /mnt/d/PanPep_Reusability/jupter/CASE3_majority_bedroc

File: /mnt/d/PanPep_Reusability/jupter/CASE3_majority_bedroc/summary_by_directory.csv
                                           Directory  BEDROC_Mean  BEDROC_Std  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...     0.225475    0.033324   

   File_Count  
0           4  


File: /mnt/d/PanPep_Reusability/jupter/CASE3_majority_bedroc/temp_ATDALMTGF_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                   Filename  BEDROC_Score  Alpha  
0  ATDALMTGF_sorted.parquet      0.225947     20  


File: /mnt/d/PanPep_Reusability/jupter/CASE3_majority_bedroc/temp_LLAGIGTVPI_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                    Filename  BEDROC_Score  Alpha  
0  LLAGIGTVPI_sorted.parquet      0.183163     20  


File: /mnt/d/PanPep_Reusab

In [ ]:
!{sys.executable} "../metric_calculation/success_rate&hit_rate.py" \
    --root_dir "./CASE3_majority_sort/CASE3_majority_rank" \
    --output "CASE3_majority_sort" \
    --output_dir "./CASE3_majority_success"

2026-03-17 15:05:15,534 - INFO - Multiprocessing start method set to 'spawn'.
2026-03-17 15:05:15,627 - INFO - Configuration: ProcessingConfig(root_dir='/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE3_majority_rank', top_k_ratio=1, batch_size=150, num_gpus=1, output_file='CASE3_majority_sort', output_dir='./CASE3_majority_success', gpu_id=None)
2026-03-17 15:05:15,628 - INFO - Detected 1 GPU(s)
Processing directories: 100%|█████████████████████| 1/1 [00:01<00:00,  1.98s/it]
2026-03-17 15:05:17,645 - INFO - Combining 1 DataFrames...
2026-03-17 15:05:17,958 - INFO - Final results saved to CASE3_majority_success/CASE3_majority_sort.csv and CASE3_majority_success/CASE3_majority_sort.parquet


In [ ]:
!{sys.executable} "../metric_calculation/get_success_AUC.py" \
    --input "./CASE3_majority_success/CASE3_majority_sort.parquet" \
    --output "CASE3_majority_success_AUC" 

Found 1 directories.
1. '/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE3_majority_rank'
['/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE3_majority_rank'] n=57265  Top_K range=(1, 57265)  y range=(0, 1)
/mnt/d/PanPep_Reusability/jupter/../metric_calculation/get_success_AUC.py:190: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = float(np.trapz(ys, xs_norm))
  AUC@0.0100%: area=2.30261e-06
  AUC@0.1000%: area=5.33609e-05
  AUC@1.0000%: area=0.000862028
  AUC@5.0000%: area=0.00814297
  AUC@10.0000%: area=0.0241858
  AUC@20.0000%: area=0.0657735
  AUC@100.0000%: area=0.68936

Saved results to: CASE3_majority_success_AUC
